# tinylm sft — analysis

Analysis-only notebook: loads the SFT checkpoints produced by `train.py`
(full-FT and, when present, LoRA) and compares them against the base pretrain
checkpoint — chat generations, per-source masked val loss, zero-shot benchmarks
(SFT shouldn't tank them). Run `train.py` (and `USE_LORA=1 train.py`) first.


In [ ]:
import os

os.environ.setdefault("HF_HOME", "/mnt/ai/data/hf")

from pathlib import Path

import torch

import train  # sft/train.py: constants, make_model, make_datamodules

DEVICE = train.DEVICE

CKPTS = {
    "base": Path(train.BASE_CHECKPOINT),
    "full-ft": train.RUN_DIR / "chimera_gpt6m_sft.pt",
    "lora-r16": train.RUN_DIR / "chimera_gpt6m_sft_lora.pt",
}
CKPTS = {k: p for k, p in CKPTS.items() if p.exists()}

dms = train.make_datamodules()
tok = dms[0].tokenizer
eos_id, bos_id = dms[0].eos_id, dms[0].bos_id
im_end_id = tok._tok.token_to_id("<|im_end|>")


def load(name):
    m = train.make_model(tok.vocab_size)
    m.load_state_dict(torch.load(CKPTS[name], map_location="cpu"))
    return m.to(DEVICE, torch.bfloat16).eval()


models = {name: load(name) for name in CKPTS}
print("loaded:", ", ".join(models))

loaded: base, full-ft, lora-r16


## Masked val loss per source

Loss over assistant tokens only, per SFT source. `base` shows where each
checkpoint started; deltas show what SFT actually moved.


In [2]:
import pandas as pd

rows = {}
for name, m in models.items():
    rows[name] = {dm.name: train.evaluate(m, dm.val_dataloader(), eos_id) for dm in dms}
pd.DataFrame(rows).round(4)

,base,full-ft,lora-r16
gooaq-chat,4.1852,3.4422,3.5887
squad-chat,7.4128,2.4630,3.0525
everyday-conversations,3.9984,2.5905,2.7175


## Chat


In [3]:
from chimera.data.chat_template import render


@torch.no_grad()
def chat(model, question, max_new_tokens=80):
    prompt = render([{"role": "user", "content": question}], add_generation_prompt=True)
    ids = tok._tok.encode(prompt, add_special_tokens=False).ids
    x = torch.tensor([[bos_id] + ids], device=DEVICE)
    out = []
    for _ in range(max_new_tokens):
        nxt = int(model(x)[0, -1].argmax())
        if nxt in (eos_id, im_end_id):
            break
        out.append(nxt)
        x = torch.cat([x, torch.tensor([[nxt]], device=DEVICE)], dim=1)
    return tok._tok.decode(out).strip()


PROMPTS = [
    "What color is the sky?",
    "Hi! How are you today?",
    "Tom has a red ball and a blue kite. He plays with them in the park every "
    "Sunday with his sister Amy.\n\nWhat color is Tom's ball?",
]

for q in PROMPTS:
    print(f"### {q[:70]}")
    for name, m in models.items():
        print(f"  {name:>8}: {chat(m, q)!r}")
    print()

### What color is the sky?
      base: 'The sky is a beautiful blue sky. It is a beautiful blue sky with a bright blue sky. It is a beautiful blue sky with a bright blue sky.\nThe sky is a beautiful blue sky with a bright blue sky. It is a beautiful blue sky with a bright blue sky.\nThe sky is a beautiful blue sky with a bright blue sky. It is a beautiful blue sky with a'
   full-ft: 'The sky is the color of the sky. It is the color of the sky. The sky is the color of the sky. The sky is the color of the sky.'
  lora-r16: 'The sky is a color that is a white color that is white and white. The sky is white and white, and the sky is white. The sky is white and white, and the sky is white. The sky is white and white, and the sky is white.'

### Hi! How are you today?
      base: "I'm not just a book; I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman. I'm a bookman."
   full-ft:

## Zero-shot benchmarks

Same task set as pretrain. The check is that SFT didn't tank the base
capabilities (catastrophic forgetting), not that it improves them.


In [4]:
from chimera.evals import ChimeraLM, TASKS, headline, run_eval

bench = {}
for name, m in models.items():
    lm = ChimeraLM(
        m,
        tok,
        eot_id=eos_id,
        bos_id=bos_id,
        block_size=m.seq_len,
        device=DEVICE,
        batch_tokens=131_072,
    )
    results = run_eval(lm, TASKS)
    bench[name] = {t: headline(results[t])[1] * 100 for t in TASKS}
pd.DataFrame(bench).round(2)

[bench] loaded pretokenized eval inputs from chimera-tokcache-a4b5d036f29b6dacbfcf841776cbcc64.pkl


loglikelihood: 100%|██████████| 156330/156330 [00:22<00:00, 6936.71it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:06<00:00, 16.24it/s]


[bench] loaded pretokenized eval inputs from chimera-tokcache-a4b5d036f29b6dacbfcf841776cbcc64.pkl


...

[bench] loaded pretokenized eval inputs from chimera-tokcache-a4b5d036f29b6dacbfcf841776cbcc64.pkl


loglikelihood: 100%|██████████| 156330/156330 [00:22<00:00, 6900.66it/s]


bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:05<00:00, 17.03it/s]


,base,full-ft,lora-r16
blimp,69.86,70.91,70.40
lambada_openai,17.47,20.47,20.84
piqa,56.47,55.82,55.88
sciq,68.10,67.30,66.50
arc_easy,35.23,34.51,34.76


Past results and run notes live in the [README](README.md).


## Playground

Free-form experimentation with the FULL sampler (`model.sample`): temperature,
top-p/top-k, min-p, repetition / frequency / presence penalties, seed. `generate()`
takes a plain string (single user turn) or a full `messages` list (multi-turn),
renders it through the canonical chat template, and returns only the assistant
continuation (sliced in token space). Tweak and re-run the last cell.


In [17]:
def generate(
    prompt_or_messages,
    model_name="full-ft",
    max_new_tokens=128,
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    min_p=0.0,
    repetition_penalty=1.1,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    seed=None,
):
    messages = (
        [{"role": "user", "content": prompt_or_messages}]
        if isinstance(prompt_or_messages, str)
        else prompt_or_messages
    )
    prompt = render(messages, add_generation_prompt=True)
    n_prompt = 1 + len(tok._tok.encode(prompt, add_special_tokens=False).ids)  # +bos
    _, generated = models[model_name].sample(
        tok,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        min_p=min_p,
        repetition_penalty=repetition_penalty,
        frequency_penalty=frequency_penalty,
        presence_penalty=presence_penalty,
        bos_token_id=bos_id,
        stop_token_ids={eos_id, im_end_id},
        seed=seed,
        return_token_ids=True,
    )
    token_ids = generated[0].tolist()  # sample returns a (1, T) tensor
    cont = [t for t in token_ids[n_prompt:] if t not in (eos_id, im_end_id)]
    return tok._tok.decode(cont).strip()

In [18]:
# sampler A/B on one prompt: greedy vs default vs hot + heavy anti-repetition
q = "What color is the sky?"
for label, kw in {
    "greedy": dict(temperature=0.0),
    "default": dict(seed=0),
    "hot+penalized": dict(
        temperature=1.0,
        repetition_penalty=1.3,
        frequency_penalty=1.1,
        presence_penalty=1.1,
        seed=0,
    ),
}.items():
    print(f"{label:>14}: {generate(q, **kw)!r}\n")

        greedy: "The sun is a bright blue, and it's a very bright yellow. The sun is light pink with white and has a red and yellow color. The sun is a bright yellow color because it's bright green and has a long-lived appearance."

       default: "Both colors are a lot of white. ... Black, blue and yellow are both common red and purple because they can be colored or yellow. It's pretty small, but has been suggested by the American Academy for more than half of our time and could also have made us black as long as we know it."

 hot+penalized: "Both colors are a lot of white. ... Black (because they're blue and dark) yellow, which can be colored or brown at some point to reddeny their lengths from black under light for grayness that aren't too close as green! They have little resembles orange because they just like pink on them – with shades similar in colour depending upon why you've left-handed it — but don's bright brighter after all..."



In [ ]:
# scratch cell — try your own prompt / knobs / model ("base", "full-ft", "lora-r16")
print(
    generate(
        [
            {"role": "user", "content": "What is the capital of France?"},
        ],
        model_name="base",
        temperature=0,
        repetition_penalty=1.1,
        seed=1,
        max_new_tokens=256,
    )
)

The capital of France was a powerful force in the United States. It was a powerful force that could destroy the country's economy and make it an ideal for the people who lived there. This power was made by the French government, which had been used to protect the country from the war. The capital of France was also known as the capital of France.

Question: What was the capital of France?
Answer: the capital of France

Question: Who was the capital of France?
Answer: the capital of France

Question: What was the capital of France?
Answer: the capital of France

Question: What was the capital of France?
Answer: the capital of France

Question: What was the capital of France?
Answer: the capital of France
